# research.ipynb — Descarga y mantenimiento de precios

**Propósito:** Única fuente de datos del sistema. Descarga OHLCV diario, dividendos y splits desde Yahoo Finance (HTTP directo), construye precios ajustados y los guarda en disco.

**Flujo:**
1. Montar Drive y definir paths
2. Leer `universe.csv` para saber qué tickers están activos
3. Para cada ticker: descargar o actualizar (append-only) el CSV crudo en `data/prices/`
4. Construir `prices_adjusted.csv` (panel de cierre ajustado, todos los tickers)
5. Verificar integridad del panel

**Output:** `data/prices/<TICKER>.csv` (crudo) + `data/prices_adjusted.csv` (procesado)  
**Pre-requisito:** Ninguno. Este notebook es el primero en correr.  
**Referencia:** MScFE SKILL §2.1 (fetch & clean), §1.2 (project structure)


In [1]:
# ── 0. MOUNT & PATHS ──────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys

BASE    = Path('/content/drive/MyDrive/portfolio_quant')
CONFIG  = BASE / 'config'
DATA    = BASE / 'data'
PRICES  = DATA / 'prices'          # CSVs crudos por ticker

# Crear carpetas si no existen
PRICES.mkdir(parents=True, exist_ok=True)
(DATA / 'signals').mkdir(exist_ok=True)
(BASE / 'live').mkdir(exist_ok=True)

print('BASE:   ', BASE.exists())
print('CONFIG: ', CONFIG.exists())
print('PRICES: ', PRICES.exists())
print('universe.csv:', (CONFIG / 'universe.csv').exists())
print('prices_adjusted.csv:', (DATA / 'prices_adjusted.csv').exists())

Mounted at /content/drive
BASE:    True
CONFIG:  True
PRICES:  True
universe.csv: True
prices_adjusted.csv: True


In [2]:
# ── 1. IMPORTS ────────────────────────────────────────────────────────────────
# Librerías estándar — sin dependencias externas al ecosistema científico Python
import requests
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime, date

print('pandas', pd.__version__, '| numpy', np.__version__)
print('Fecha actual:', date.today())

pandas 2.2.2 | numpy 2.0.2
Fecha actual: 2026-06-18


In [3]:
# ── 2. LEER UNIVERSO ──────────────────────────────────────────────────────────
# universe.csv es la fuente de verdad de qué tickers están activos.
# Este notebook SOLO LEE el archivo; los cambios al universo se hacen manualmente.
universe_df = pd.read_csv(CONFIG / 'universe.csv')

# Filtrar activos con active == 'yes'
active_tickers = universe_df[universe_df['active'] == 'yes']['ticker'].tolist()
print(f'{len(active_tickers)} tickers activos: {active_tickers}')

# Tickers que ya tienen CSV descargado
existing_csvs = sorted(f.stem for f in PRICES.glob('*.csv'))
new_tickers   = [t for t in active_tickers if t not in existing_csvs]
print(f'\nYa descargados ({len(existing_csvs)}): {existing_csvs}')
print(f'Nuevos a descargar ({len(new_tickers)}): {new_tickers}')

24 tickers activos: ['SHY', 'BOXX', 'TLT', 'IEF', 'AGG', 'LQD', 'HYG', 'EMB', 'TIP', 'PFF', 'SPY', 'EFA', 'VEA', 'EEM', 'VNQ', 'GLD', 'SLV', 'USO', 'DBC', 'DBMF', 'BTC-USD', 'VT', 'RSST', 'RSSB']

Ya descargados (24): ['AGG', 'BOXX', 'BTC-USD', 'DBC', 'DBMF', 'EEM', 'EFA', 'EMB', 'GLD', 'HYG', 'IEF', 'LQD', 'PFF', 'RSSB', 'RSST', 'SHY', 'SLV', 'SPY', 'TIP', 'TLT', 'USO', 'VEA', 'VNQ', 'VT']
Nuevos a descargar (0): []


In [4]:
# ── 3. FUNCIÓN DE DESCARGA (Yahoo Finance HTTP directo) ───────────────────────
# Sin yfinance package — HTTP puro para independencia del proveedor.
# Para cambiar de proveedor: reemplazar solo esta función.
# Output siempre: DataFrame con columnas [date, open, high, low, close, volume, dividends, splits]

def fetch_ohlcv_yahoo(ticker: str, range_: str = '10y', interval: str = '1d') -> pd.DataFrame:
    """
    Descarga OHLCV + eventos corporativos desde Yahoo Finance v8 API.

    Args:
        ticker  (str): Símbolo Yahoo Finance (e.g. 'SPY', 'RSSB').
        range_  (str): Ventana histórica. Default '10y'.
        interval(str): Frecuencia. Default '1d' (diaria).

    Returns:
        pd.DataFrame: Columnas [date, open, high, low, close, volume, dividends, splits].
                      'date' es datetime normalizado (sin hora).

    Note:
        Retorna DataFrame vacío si el ticker no existe o falla la descarga.
        Para cambiar de proveedor: mantener el mismo schema de columnas de salida.
    """
    url = f'https://query1.finance.yahoo.com/v8/finance/chart/{ticker}'
    params = {'range': range_, 'interval': interval, 'events': 'div,splits'}
    headers = {'User-Agent': 'Mozilla/5.0'}

    try:
        r = requests.get(url, params=params, headers=headers, timeout=30)
        r.raise_for_status()
        res = r.json()['chart']['result'][0]
    except Exception as e:
        print(f'  [ERROR] {ticker}: {e}')
        return pd.DataFrame()

    timestamps = res['timestamp']
    quote      = res['indicators']['quote'][0]

    dates_idx = pd.to_datetime(timestamps, unit='s').normalize()

    df = pd.DataFrame({
        'date':      dates_idx,
        'open':      quote.get('open',   [None]*len(timestamps)),
        'high':      quote.get('high',   [None]*len(timestamps)),
        'low':       quote.get('low',    [None]*len(timestamps)),
        'close':     quote.get('close',  [None]*len(timestamps)),
        'volume':    quote.get('volume', [0]*len(timestamps)),
        'dividends': 0.0,
        'splits':    0.0,
    })

    # Inyectar dividendos
    events = res.get('events', {})
    if 'dividends' in events:
        for ev in events['dividends'].values():
            ev_date = pd.Timestamp(ev['date'], unit='s').normalize()
            mask = df['date'] == ev_date
            df.loc[mask, 'dividends'] = ev.get('amount', 0.0)

    # Inyectar splits
    if 'splits' in events:
        for ev in events['splits'].values():
            ev_date = pd.Timestamp(ev['date'], unit='s').normalize()
            mask = df['date'] == ev_date
            df.loc[mask, 'splits'] = ev.get('splitRatio', 0.0)

    df = df.dropna(subset=['close']).set_index('date').sort_index()
    return df


# Smoke test rápido
test_df = fetch_ohlcv_yahoo('SPY', range_='5d')
print('Smoke test SPY:', test_df.shape)
print(test_df.tail(2).round(2))

Smoke test SPY: (5, 7)
              open    high     low   close    volume  dividends  splits
date                                                                   
2026-06-17  751.29  752.15  739.22  740.96  85718600        0.0     0.0
2026-06-18  747.76  748.18  743.86  746.60  48704516        0.0     0.0


In [5]:
# ── 4. DESCARGA / ACTUALIZACIÓN INCREMENTAL ───────────────────────────────────
# Para cada ticker activo:
#   - Si no existe CSV: descarga completa (10y)
#   - Si ya existe: descarga solo los últimos 5 días y hace append de filas nuevas
# Resultado: un CSV por ticker en data/prices/, append-only.

SLEEP_SECONDS = 0.8   # pausa entre requests para no ser bloqueado

def load_raw_csv(ticker: str) -> pd.DataFrame:
    """Lee el CSV crudo de un ticker. Retorna DataFrame vacío si no existe."""
    csv_path = PRICES / f'{ticker}.csv'
    if not csv_path.exists():
        return pd.DataFrame()
    return pd.read_csv(csv_path, parse_dates=['date'], index_col='date').sort_index()

def save_raw_csv(ticker: str, df: pd.DataFrame) -> None:
    """Guarda el CSV crudo de un ticker (sobrescribe completo)."""
    df.to_csv(PRICES / f'{ticker}.csv')

def update_ticker(ticker: str) -> dict:
    """
    Descarga o actualiza el CSV crudo de un ticker.

    Returns:
        dict: {'ticker': str, 'rows_before': int, 'rows_after': int, 'new_rows': int}
    """
    existing    = load_raw_csv(ticker)
    rows_before = len(existing)

    if existing.empty:
        df_new = fetch_ohlcv_yahoo(ticker, range_='10y')
        if df_new.empty:
            return {'ticker': ticker, 'rows_before': 0, 'rows_after': 0, 'new_rows': 0, 'status': 'ERROR'}
        save_raw_csv(ticker, df_new)
        return {'ticker': ticker, 'rows_before': 0, 'rows_after': len(df_new), 'new_rows': len(df_new), 'status': 'FULL'}
    else:
        df_recent = fetch_ohlcv_yahoo(ticker, range_='5d')
        if df_recent.empty:
            return {'ticker': ticker, 'rows_before': rows_before, 'rows_after': rows_before, 'new_rows': 0, 'status': 'NO_DATA'}

        last_date  = existing.index.max()
        df_append  = df_recent[df_recent.index > last_date]

        if df_append.empty:
            return {'ticker': ticker, 'rows_before': rows_before, 'rows_after': rows_before, 'new_rows': 0, 'status': 'UP_TO_DATE'}

        df_combined = pd.concat([existing, df_append]).sort_index()
        save_raw_csv(ticker, df_combined)
        return {'ticker': ticker, 'rows_before': rows_before, 'rows_after': len(df_combined), 'new_rows': len(df_append), 'status': 'UPDATED'}


# Correr actualización para todos los tickers activos
results = []
for ticker in active_tickers:
    result = update_ticker(ticker)
    results.append(result)
    status = result['status']
    new    = result['new_rows']
    after  = result['rows_after']
    print(f'  {ticker:8s} [{status:10s}] +{new:3d} filas → total {after:4d}')
    time.sleep(SLEEP_SECONDS)

# Resumen
results_df = pd.DataFrame(results)
print('\n--- RESUMEN ---')
print(results_df.groupby('status')['ticker'].count().to_string())

  SHY      [UPDATED   ] +  2 filas → total 2515
  BOXX     [UPDATED   ] +  2 filas → total  871
  TLT      [UPDATED   ] +  2 filas → total 2515
  IEF      [UPDATED   ] +  2 filas → total 2515
  AGG      [UPDATED   ] +  2 filas → total 2515
  LQD      [UPDATED   ] +  2 filas → total 2515
  HYG      [UPDATED   ] +  2 filas → total 2515
  EMB      [UPDATED   ] +  2 filas → total 2515
  TIP      [UPDATED   ] +  2 filas → total 2515
  PFF      [UPDATED   ] +  2 filas → total 2515
  SPY      [UPDATED   ] +  2 filas → total 2515
  EFA      [UPDATED   ] +  2 filas → total 2515
  VEA      [UPDATED   ] +  2 filas → total 2515
  EEM      [UPDATED   ] +  2 filas → total 2515
  VNQ      [UPDATED   ] +  2 filas → total 2515
  GLD      [UPDATED   ] +  2 filas → total 2515
  SLV      [UPDATED   ] +  2 filas → total 2515
  USO      [UPDATED   ] +  2 filas → total 2515
  DBC      [UPDATED   ] +  2 filas → total 2515
  DBMF     [UPDATED   ] +  2 filas → total 1789
  BTC-USD  [UPDATED   ] +  1 filas → tot

In [6]:
# ── 5. CONSTRUCCIÓN DE PRECIOS AJUSTADOS ──────────────────────────────────────
# Ajuste hacia atrás por splits y dividendos (método CRSP / Yahoo Adj Close).
# El CSV crudo NO se modifica — el ajuste se recalcula en cada corrida.
# Ref: MScFE SKILL §2.1 (compute_returns, data workflows)

def compute_adjusted_close(df_raw: pd.DataFrame) -> pd.Series:
    """
    Calcula el precio ajustado hacia atrás desde datos crudos + eventos corporativos.

    Args:
        df_raw (pd.DataFrame): DataFrame con columnas [close, dividends, splits].

    Returns:
        pd.Series: Serie de precios ajustados (mismo índice que df_raw).
    """
    price_close   = df_raw['close'].astype(float).copy()
    dividends_raw = df_raw.get('dividends', pd.Series(0.0, index=price_close.index)).fillna(0.0)
    splits_raw    = df_raw.get('splits',    pd.Series(0.0, index=price_close.index)).fillna(0.0)

    split_factor     = splits_raw.replace(0, np.nan)
    cum_split_factor = split_factor[::-1].fillna(1).cumprod()[::-1].shift(-1).fillna(1.0)

    div_ratio        = (1 - dividends_raw / price_close).clip(lower=0.0001)
    cum_div_factor   = div_ratio[::-1].cumprod()[::-1].shift(-1).fillna(1.0)

    price_adjusted = price_close * cum_split_factor * cum_div_factor
    return price_adjusted


# Construir panel de precios ajustados
tickers_on_disk = sorted(f.stem for f in PRICES.glob('*.csv'))
prices_adjusted_dict = {}

for ticker in tickers_on_disk:
    df_raw = load_raw_csv(ticker)
    if df_raw.empty or 'close' not in df_raw.columns:
        print(f'  [SKIP] {ticker}: CSV vacío o sin columna close')
        continue
    prices_adjusted_dict[ticker] = compute_adjusted_close(df_raw)

prices_adjusted_df = pd.DataFrame(prices_adjusted_dict).sort_index()

output_path = DATA / 'prices_adjusted.csv'
prices_adjusted_df.to_csv(output_path)

print(f'Panel ajustado: {prices_adjusted_df.shape}')
print(f'Período: {prices_adjusted_df.index.min().date()} → {prices_adjusted_df.index.max().date()}')
print(f'Guardado en: {output_path}')
print('\nÚltimas 3 filas:')
print(prices_adjusted_df.tail(3).round(2).to_string())

Panel ajustado: (3654, 24)
Período: 2016-06-17 → 2026-06-18
Guardado en: /content/drive/MyDrive/portfolio_quant/data/prices_adjusted.csv

Últimas 3 filas:
              AGG    BOXX   BTC-USD    DBC   DBMF    EEM     EFA    EMB     GLD    HYG    IEF     LQD    PFF   RSSB   RSST    SHY    SLV     SPY     TIP    TLT     USO    VEA    VNQ      VT
date                                                                                                                                                                                         
2026-06-16  98.97  117.03       NaN  27.90  30.63  68.64  104.31  96.67  397.63  80.03  94.52  109.12  31.19  31.02  32.75  82.12  63.39  750.33  109.76  86.19  115.47  72.32  98.06  157.99
2026-06-17  98.61  117.00  65838.23  27.71  30.67  68.56  103.78  96.28  388.60  79.73  94.02  108.77  31.09  30.56  32.39  81.88  60.61  740.96  109.03  86.33  114.23  72.00  95.61  156.41
2026-06-18  98.98  117.05  62609.28  27.53  30.84  70.79  104.44  96.79  388.16  79.9

In [7]:
# ── 6. VERIFICACIÓN DE INTEGRIDAD ─────────────────────────────────────────────
# Ref: MScFE SKILL §10 (debugging checklist)

prices_adj = pd.read_csv(DATA / 'prices_adjusted.csv', parse_dates=['date'], index_col='date').sort_index()

print('=== VERIFICACIÓN DEL PANEL ===')
print(f'Shape: {prices_adj.shape}')
print(f'Período: {prices_adj.index.min().date()} → {prices_adj.index.max().date()}')

nan_count = prices_adj.isna().sum()
tickers_with_nans = nan_count[nan_count > 0]
if tickers_with_nans.empty:
    print('✓ Sin NaNs en el panel')
else:
    print(f'⚠ Tickers con NaNs:')
    print(tickers_with_nans.to_string())

system_tickers = universe_df[universe_df['active'] == 'yes']['ticker'].tolist()
in_panel = [t for t in system_tickers if t in prices_adj.columns]
missing  = [t for t in system_tickers if t not in prices_adj.columns]
print(f'\n✓ Tickers del sistema en panel ({len(in_panel)}): {in_panel}')
if missing:
    print(f'⚠ Faltantes en panel ({len(missing)}): {missing}')

returns_daily = prices_adj.pct_change()
extreme_mask  = (returns_daily.abs() > 0.50).any()
extreme_tickers = extreme_mask[extreme_mask].index.tolist()
if extreme_tickers:
    print(f'\n⚠ Retornos >50% detectados (revisar splits): {extreme_tickers}')
else:
    print('✓ Sin retornos diarios extremos (>50%)')

print('\n--- Cobertura por ticker (sistema activo) ---')
coverage = pd.DataFrame({
    'primer_fecha': prices_adj[in_panel].apply(lambda col: col.dropna().index.min()),
    'ultima_fecha':  prices_adj[in_panel].apply(lambda col: col.dropna().index.max()),
    'n_obs':         prices_adj[in_panel].count(),
    'nans':          prices_adj[in_panel].isna().sum(),
})
print(coverage.to_string())

=== VERIFICACIÓN DEL PANEL ===
Shape: (3654, 24)
Período: 2016-06-17 → 2026-06-18
⚠ Tickers con NaNs:
AGG        1139
BOXX       2783
BTC-USD       1
DBC        1139
DBMF       1865
EEM        1139
EFA        1139
EMB        1139
GLD        1139
HYG        1139
IEF        1139
LQD        1139
PFF        1139
RSSB       3018
RSST       2955
SHY        1139
SLV        1139
SPY        1139
TIP        1139
TLT        1139
USO        1139
VEA        1139
VNQ        1139
VT         1139

✓ Tickers del sistema en panel (24): ['SHY', 'BOXX', 'TLT', 'IEF', 'AGG', 'LQD', 'HYG', 'EMB', 'TIP', 'PFF', 'SPY', 'EFA', 'VEA', 'EEM', 'VNQ', 'GLD', 'SLV', 'USO', 'DBC', 'DBMF', 'BTC-USD', 'VT', 'RSST', 'RSSB']

⚠ Retornos >50% detectados (revisar splits): ['USO']

--- Cobertura por ticker (sistema activo) ---
        primer_fecha ultima_fecha  n_obs  nans
SHY       2016-06-17   2026-06-18   2515  1139
BOXX      2022-12-28   2026-06-18    871  2783
TLT       2016-06-17   2026-06-18   2515  1139
IEF       2

/tmp/ipykernel_16882/1925781622.py:25: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns_daily = prices_adj.pct_change()
